In [1]:
from pathlib import Path
from contextlib import closing
from datetime import datetime, timezone
import hashlib
import html
import json
import re
import shutil
import sqlite3
import urllib.request
import xml.etree.ElementTree as ET
import zipfile

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW, PROCESSED, SQL = ROOT / "data/raw", ROOT / "data/processed", ROOT / "sql"
DB = ROOT / "data/coffee.db"
for directory in [RAW / "faostat", RAW / "worldbank", RAW / "ico", RAW / "reference", PROCESSED]:
    directory.mkdir(parents=True, exist_ok=True)
START, END = 2000, 2023
pd.set_option("display.max_rows", 80)

In [2]:
WORLD_BANK_TERMS = {"licence": "CC-BY-4.0; World Bank Terms of Use for Datasets",
                    "licence_url": "https://www.worldbank.org/en/about/legal/terms-of-use-for-datasets"}
sources = {
    "faostat_trade": {
        "url": "https://bulks-faostat.fao.org/production/Trade_CropsLivestock_E_All_Data_(Normalized).zip",
        "path": "faostat/Trade_CropsLivestock_E_All_Data_(Normalized).zip",
        "documentation": "https://www.fao.org/faostat/en/#data/TCL",
        "licence": "CC-BY-4.0; FAO Statistical Database Terms of Use",
        "licence_url": "https://www.fao.org/contact-us/terms/db-terms-of-use/en/",
        "note": "Not in Git (over the GitHub 100 MB limit); green-coffee extract: data/processed/faostat_trade_coffee.csv",
    },
    "pink_sheet": {
        # The World Bank changes this URL with every edition; take the current one from the "Annual prices" link.
        "url": "https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Annual.xlsx",
        "path": "worldbank/CMO-Historical-Data-Annual.xlsx",
        "documentation": "https://www.worldbank.org/en/research/commodity-markets",
        **WORLD_BANK_TERMS,
        "units": "USD/kg, nominal",
    },
    "cpi_usa": {
        "url": f"https://api.worldbank.org/v2/country/USA/indicator/FP.CPI.TOTL?format=json&date={START}:{END}&per_page=100",
        "path": f"worldbank/cpi_usa_{START}_{END}.json",
        "documentation": "https://data.worldbank.org/indicator/FP.CPI.TOTL?locations=US",
        **WORLD_BANK_TERMS,
        "units": "index, 2010 = 100",
    },
    "ico_production": {
        # www.ico.org no longer serves the historical files; the legacy host still does, over plain HTTP.
        "url": "http://dev.ico.org/historical/1990%20onwards/Excel/1a%20-%20Total%20production.xlsx",
        "path": "ico/1a-total-production.xlsx",
        "documentation": "https://ico.org/resources/historical-data-on-the-global-coffee-trade/",
        "licence": "ICO historical data, free download; attribution: International Coffee Organization",
        "purpose": "Coffee type per exporting country: (A) Arabica, (R) Robusta, (A/R) mainly Arabica, (R/A) mainly Robusta",
    },
    "weather_optimum": {
        "path": "reference/weather_optimum.csv",
        "documentation": "https://doi.org/10.1590/S1677-04202006000100006",
        "attribution": "DaMatta F.M., Ramalho J.D.C. (2006). Braz. J. Plant Physiol. 18(1): 55-81",
        "purpose": "Optimal mean annual temperature and annual rainfall per species; maintained by hand",
    },
    "species_overrides": {
        "path": "reference/species_overrides.csv",
        "documentation": "https://apps.fas.usda.gov/psdonline/circulars/2024/12/coffee.pdf",
        "attribution": "USDA Foreign Agricultural Service (2024). Coffee: World Markets and Trade, December 2024",
        "purpose": "Dominant species for countries missing from the ICO list; maintained by hand",
    },
}
manifest_path = RAW / "sources.json"
previous = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
for name, source in sources.items():
    path = RAW / source["path"]
    downloaded = not path.exists()
    if downloaded:
        assert "url" in source, f"{path} is maintained by hand"
        request = urllib.request.Request(source["url"], headers={"User-Agent": "Mozilla/5.0", "Accept": "*/*"})
        temporary = path.with_suffix(path.suffix + ".part")
        with urllib.request.urlopen(request, timeout=300) as response, temporary.open("wb") as file:
            shutil.copyfileobj(response, file)
        temporary.replace(path)
    source["downloaded_at_utc"] = (datetime.now(timezone.utc).isoformat() if downloaded
                                   else previous.get(name, {}).get("downloaded_at_utc", datetime.now(timezone.utc).isoformat()))
    source["bytes"] = path.stat().st_size
    with path.open("rb") as file:
        source["sha256"] = hashlib.file_digest(file, "sha256").hexdigest()

assert zipfile.is_zipfile(RAW / sources["faostat_trade"]["path"])
catalog = ET.parse(RAW / "faostat/datasets_E.xml").getroot()
sources["faostat_trade"]["last_updated"] = next(d.findtext("DateUpdate") for d in catalog if d.findtext("DatasetCode") == "TCL")
pink_sheet_header = pd.read_excel(RAW / sources["pink_sheet"]["path"], "Annual Prices (Nominal)", header=None, nrows=4)[0]
sources["pink_sheet"]["last_updated"] = pd.to_datetime(pink_sheet_header.iloc[3].removeprefix("Updated on ")).date().isoformat()
cpi_raw = json.loads((RAW / sources["cpi_usa"]["path"]).read_text(encoding="utf-8"))
sources["cpi_usa"]["last_updated"] = cpi_raw[0]["lastupdated"]
manifest_path.write_text(json.dumps(previous | sources, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(pd.DataFrame(sources).T[["path", "bytes", "downloaded_at_utc", "last_updated"]])

,path,bytes,downloaded_at_utc,last_updated
faostat_trade,faostat/Trade_CropsLivestock_E_All_Data_(Norma...,273514210,2026-11-16T14:17:48.853976+00:00,2026-07-24T00:00:00
pink_sheet,worldbank/CMO-Historical-Data-Annual.xlsx,3176359,2026-11-16T14:17:49.145155+00:00,2026-09-02
cpi_usa,worldbank/cpi_usa_2000_2023.json,5500,2026-11-16T14:17:49.429363+00:00,2026-11-11
ico_production,ico/1a-total-production.xlsx,36886,2026-11-16T14:17:49.430362+00:00,NaN
weather_optimum,reference/weather_optimum.csv,383,2026-11-16T14:17:49.430362+00:00,NaN
species_overrides,reference/species_overrides.csv,283,2026-11-16T15:27:02.978325+00:00,NaN


In [3]:
TRADE_UNITS = {5910: "t", 5922: "1000 USD"}  # Export quantity, Export value
with zipfile.ZipFile(RAW / sources["faostat_trade"]["path"]) as archive:
    with archive.open("Trade_CropsLivestock_E_All_Data_(Normalized).csv") as stream:
        trade = pd.concat(
            [chunk.loc[chunk["Item Code"].eq(656) & chunk["Element Code"].isin(TRADE_UNITS)]
             for chunk in pd.read_csv(stream, chunksize=1_000_000, low_memory=False,
                                      usecols=["Area Code", "Area Code (M49)", "Area", "Item Code", "Item",
                                               "Element Code", "Element", "Year", "Unit", "Value", "Flag"],
                                      dtype={"Area Code (M49)": "string", "Flag": "string"})],
            ignore_index=True,
        )
assert trade["Item"].eq("Coffee, green").all()
assert trade["Unit"].eq(trade["Element Code"].map(TRADE_UNITS)).all()
assert not trade.duplicated(["Area Code", "Element Code", "Year"]).any()
trade = trade.drop(columns=["Item Code", "Item"])
trade.to_csv(PROCESSED / "faostat_trade_coffee.csv", index=False)
display(trade.groupby(["Element", "Unit"]).agg(rows=("Value", "size"), missing=("Value", lambda s: s.isna().sum()),
                                               areas=("Area", "nunique"), first_year=("Year", "min"), last_year=("Year", "max")))

,,rows,missing,areas,first_year,last_year
Element,Unit,,,,,
Export quantity,t,12590,0,272,1961,2024
Export value,1000 USD,12629,0,272,1961,2024


In [4]:
prices = pd.read_excel(RAW / sources["pink_sheet"]["path"], "Annual Prices (Nominal)", header=6, skiprows=[7])
prices = prices.rename(columns={prices.columns[0]: "year", "Coffee, Arabica": "arabica_usd_kg", "Coffee, Robusta": "robusta_usd_kg"})
prices = prices.loc[prices["year"].between(START, END), ["year", "arabica_usd_kg", "robusta_usd_kg"]]
assert prices["year"].tolist() == list(range(START, END + 1))
assert prices[["arabica_usd_kg", "robusta_usd_kg"]].gt(0).all().all()

assert all(row["indicator"]["id"] == "FP.CPI.TOTL" and row["countryiso3code"] == "USA" for row in cpi_raw[1])
cpi = pd.DataFrame({"year": [int(row["date"]) for row in cpi_raw[1]],
                    "cpi_usa": [row["value"] for row in cpi_raw[1]]}).sort_values("year", ignore_index=True)
assert cpi["year"].tolist() == list(range(START, END + 1)) and cpi["cpi_usa"].notna().all()
display(prices.merge(cpi, on="year", validate="one_to_one").iloc[[0, -1]])

,year,arabica_usd_kg,robusta_usd_kg,cpi_usa
0,2000,1.92,0.91,78.970721
23,2023,4.54,2.63,139.735794


In [5]:
ico = pd.read_excel(RAW / sources["ico_production"]["path"], header=None, usecols=[0, 1], names=["ico_name", "ico_type"])
ico = ico.loc[ico["ico_type"].astype(str).str.fullmatch(r"\((A|R|A/R|R/A)\)")].copy()
ico["ico_type"] = ico["ico_type"].str.strip("()")
ico["species"] = ico["ico_type"].str[0].map({"A": "arabica", "R": "robusta"})
ico["species_source"] = "ICO"
countries = pd.read_csv(PROCESSED / "countries.csv")
ico_aliases = {"Democratic Republic of Congo": "COD", "Tanzania": "TZA", "Trinidad & Tobago": "TTO", "Venezuela": "VEN"}
ico["iso3"] = ico["ico_name"].map(countries.set_index("country")["iso3"]).fillna(ico["ico_name"].map(ico_aliases))
assert ico["iso3"].notna().all(), ico.loc[ico["iso3"].isna(), "ico_name"].tolist()
species = countries.merge(ico[["iso3", "ico_name", "ico_type", "species", "species_source"]],
                          on="iso3", how="left", validate="one_to_one").set_index("iso3")
overrides = pd.read_csv(RAW / sources["species_overrides"]["path"]).set_index("iso3")
assert species.loc[overrides.index, "species"].isna().all(), "overrides only fill countries missing from ICO"
species.update(overrides)
species = species.reset_index()
species.to_csv(PROCESSED / "dominant_species.csv", index=False)

optimum = pd.read_csv(RAW / sources["weather_optimum"]["path"])
assert set(optimum["species"]) == set(species["species"].dropna())
assert optimum["temperature_min_c"].lt(optimum["temperature_max_c"]).all()
assert optimum["precipitation_min_mm"].lt(optimum["precipitation_max_mm"]).all()
display(species.groupby(["species", "ico_type"], dropna=False).size().rename("countries").reset_index())
display(optimum)
print("Producenci bez gatunku:", "; ".join(species.loc[species["species"].isna(), "country"]))

c:\Users\chame\studia\sem7\systemy-analizy-biznesowej\projekt\systemy-analizy-biznesowej-projekt\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,species,ico_type,countries
0,arabica,A,21
1,arabica,A/R,8
2,arabica,NaN,1
3,robusta,R,15
4,robusta,R/A,10
5,robusta,NaN,1
6,NaN,NaN,19


,species,temperature_min_c,temperature_max_c,precipitation_min_mm,precipitation_max_mm,source
0,arabica,18,21,1200,1800,DaMatta & Ramalho (2006): temperature after Al...
1,robusta,22,26,1200,1800,DaMatta & Ramalho (2006): temperature after Ma...


Producenci bez gatunku: Belize; Benin; Cambodia; Comoros; Dominica; Fiji; French Polynesia; Mozambique; Myanmar; New Caledonia; Paraguay; Puerto Rico; Saint Vincent and the Grenadines; Samoa; Sao Tome and Principe; Suriname; Tonga; United States of America; Vanuatu


In [6]:
m49_html = (RAW / "un/m49_overview.html").read_text(encoding="utf-8")
table = re.search(r'<table\b[^>]*id\s*=\s*"downloadTableEN"[^>]*>(.*?)</table>', m49_html, re.S).group(1)
table = re.sub(r"<!--.*?-->", "", table, flags=re.S)
m49_rows = [[html.unescape(re.sub(r"<[^>]+>", "", cell)).strip() for cell in re.findall(r"<td\b[^>]*>(.*?)</td>", row, re.S)]
            for row in re.findall(r"<tr\b[^>]*>(.*?)</tr>", table, re.S)]
m49 = pd.DataFrame([{"m49": cells[9], "iso3": cells[11], "region": cells[3], "subregion": cells[7] or cells[5]}
                    for cells in m49_rows if len(cells) >= 12 and re.fullmatch(r"[A-Z]{3}", cells[11])])
assert m49["m49"].is_unique and m49["iso3"].is_unique

# Inner join drops areas without an ISO3 code, e.g. the FAO "China" aggregate (M49 159).
export = (trade.loc[trade["Area Code"].lt(5000) & trade["Year"].between(START, END)]
          .assign(m49=lambda d: d["Area Code (M49)"].str.lstrip("'"))
          .merge(m49[["m49", "iso3"]], on="m49", validate="many_to_one"))
export = export.loc[export["iso3"].isin(countries["iso3"])]
export = export.pivot(index=["iso3", "Year"], columns="Element Code", values=["Value", "Flag"])
export.columns = [{("Value", 5910): "export_t", ("Value", 5922): "export_value_usd",
                   ("Flag", 5910): "export_t_flag", ("Flag", 5922): "export_value_flag"}[column] for column in export.columns]
export = export.astype({"export_t": float, "export_value_usd": float})
export["export_value_usd"] *= 1000
export = export.reindex(pd.MultiIndex.from_product([countries["iso3"], range(START, END + 1)], names=["iso3", "year"]))

export_years = export[["export_t", "export_value_usd"]].notna().all(axis=1).groupby("iso3").sum()
complete_iso3 = export_years.index[export_years.eq(END - START + 1)]
world_export = trade.loc[trade["Area"].eq("World") & trade["Element Code"].eq(5910)].set_index("Year")["Value"]
max_share = (export["export_t"] / export.index.get_level_values("year").map(world_export).to_numpy() * 100).groupby("iso3").max()
excluded = (countries.loc[~countries["iso3"].isin(complete_iso3)]
            .assign(export_years=lambda d: d["iso3"].map(export_years), max_world_export_share_pct=lambda d: d["iso3"].map(max_share))
            .sort_values("max_world_export_share_pct", ascending=False))
# Stop for review if an incomplete country accounts for at least 1% of world exports in any year.
assert excluded["max_world_export_share_pct"].fillna(0).lt(1).all(), excluded
excluded.to_csv(PROCESSED / "excluded_export_countries.csv", index=False)
display(pd.Series({"producers": len(countries), "producers_with_any_export": int(export_years.gt(0).sum()),
                   "complete_24_years": len(complete_iso3), "complete_pct": round(len(complete_iso3) / len(countries) * 100, 1),
                   "complete_share_of_world_export_2023_pct": round(export.xs(END, level="year").loc[complete_iso3, "export_t"].sum() / world_export[END] * 100, 1)}))
display(excluded)

producers                                  75.0
producers_with_any_export                  74.0
complete_24_years                          51.0
complete_pct                               68.0
complete_share_of_world_export_2023_pct    89.4
dtype: float64

,country,iso3,export_years,max_world_export_share_pct
70,Venezuela (Bolivarian Republic of),VEN,18,2.388180e-01
8,Central African Republic,CAF,19,2.242237e-01
12,Congo,COG,17,1.560061e-01
26,Ghana,GHA,22,9.832454e-02
0,Angola,AGO,23,4.910008e-02
21,Equatorial Guinea,GNQ,21,1.489505e-02
37,Liberia,LBR,21,1.107686e-02
1,Belize,BLZ,19,5.592867e-03
50,Paraguay,PRY,20,4.655415e-03
17,Dominica,DMA,21,3.298898e-03


In [7]:
FACT_COLUMNS = ["iso3", "year", "area_ha", "production_t", "yield_kg_ha", "yield_deviation_pct",
                "export_t", "export_value_usd", "temperature_c", "precipitation_mm",
                "temperature_anomaly_c", "precipitation_anomaly_pct",
                "area_flag", "production_flag", "yield_flag", "export_t_flag", "export_value_flag"]
coffee_climate = pd.read_csv(PROCESSED / "coffee_climate.csv")
fact = (coffee_climate.loc[coffee_climate["iso3"].isin(complete_iso3)]
        .merge(export.loc[complete_iso3].reset_index(), on=["iso3", "year"], validate="one_to_one")[FACT_COLUMNS])
assert len(fact) == len(complete_iso3) * (END - START + 1)

climate_annual = pd.read_csv(PROCESSED / "climate_annual.csv")
weather_norm = (climate_annual.loc[climate_annual["year"].between(1991, 2020)].groupby("iso3")
                .agg(temperature_norm_sd_c=("temperature_anomaly_c", "std"),
                     precipitation_norm_sd_pct=("precipitation_anomaly_pct", "std"))
                .reset_index())
dim_country = (countries.loc[countries["iso3"].isin(complete_iso3)]
               .merge(m49, on="iso3", validate="one_to_one")
               .merge(species[["iso3", "ico_type", "species", "species_source"]], on="iso3", validate="one_to_one")
               .merge(weather_norm, on="iso3", validate="one_to_one"))
dim_year = prices.merge(cpi, on="year", validate="one_to_one")
dim_year["to_usd_2023"] = dim_year.loc[dim_year["year"].eq(END), "cpi_usa"].item() / dim_year["cpi_usa"]
correlation = pd.read_csv(PROCESSED / "climate_correlations.csv").drop(columns="country")
correlation = correlation.loc[correlation["iso3"].isin(complete_iso3)]

with closing(sqlite3.connect(DB)) as con:
    con.execute("PRAGMA foreign_keys = ON")
    con.executescript((SQL / "schema.sql").read_text(encoding="utf-8"))
    for name, frame in [("ref_weather_optimum", optimum), ("dim_year", dim_year), ("dim_country", dim_country),
                        ("fact_country_year", fact), ("weather_yield_correlation", correlation)]:
        frame.to_sql(name, con, if_exists="append", index=False)
    con.executescript((SQL / "views.sql").read_text(encoding="utf-8"))
    con.commit()

In [8]:
with closing(sqlite3.connect(DB)) as con:
    assert con.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
    assert not con.execute("PRAGMA foreign_key_check").fetchall()
    rows = {name: con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
            for name in ["dim_country", "dim_year", "ref_weather_optimum", "fact_country_year",
                         "weather_yield_correlation", "v_kpi_country_year", "v_kpi_country"]}
    kpi = pd.read_sql("SELECT * FROM v_kpi_country_year ORDER BY iso3, year", con)
    kpi_country = pd.read_sql("SELECT * FROM v_kpi_country ORDER BY mean_export_t DESC", con)
countries_n, years_n = len(complete_iso3), END - START + 1
assert rows == {"dim_country": countries_n, "dim_year": years_n, "ref_weather_optimum": 2, "fact_country_year": countries_n * years_n,
                "weather_yield_correlation": countries_n, "v_kpi_country_year": countries_n * years_n, "v_kpi_country": countries_n}
assert kpi.loc[kpi["year"].eq(START), ["k4_export_t_yoy_pct", "k5_export_value_yoy_pct"]].isna().all().all()
assert kpi.loc[kpi["year"].eq(END), "k6_usd_2023_per_kg"].sub(kpi.loc[kpi["year"].eq(END), "k3_usd_per_kg"]).abs().max() < 1e-9
assert kpi.loc[kpi["k1_export_t"].eq(0), "k3_usd_per_kg"].isna().all()
assert kpi.loc[kpi["species"].isna(), "weather_in_optimum"].isna().all()
value_change = kpi.groupby("iso3")["k2_export_value_usd"].diff()
effects = kpi["value_change_volume_effect_usd"] + kpi["value_change_price_effect_usd"]
assert ((effects - value_change).abs() <= 1e-6 * value_change.abs().clip(lower=1))[effects.notna()].all()
assert kpi_country.loc[kpi_country["weak_harvest_years"].eq(0), "k10_export_resilience_pct"].isna().all()
display(pd.Series(rows, name="rows"))
display(kpi.loc[kpi["year"].eq(END) & kpi["iso3"].isin(["BRA", "VNM", "COL", "ETH"]),
                ["country", "k1_export_t", "k3_usd_per_kg", "reference_usd_kg", "k4_export_t_yoy_pct", "weather_in_norm"]])
display(kpi_country.head(10))
print(f"Baza: {DB.relative_to(ROOT)}, {DB.stat().st_size / 1024:.0f} KB")

dim_country                    51
dim_year                       24
ref_weather_optimum             2
fact_country_year            1224
weather_yield_correlation      51
v_kpi_country_year           1224
v_kpi_country                  51
Name: rows, dtype: int64

,country,k1_export_t,k3_usd_per_kg,reference_usd_kg,k4_export_t_yoy_pct,weather_in_norm
71,Brazil,2116567.93,3.456240,4.54,-0.726749,0
191,Colombia,581350.58,4.856803,4.54,-7.686475,0
311,Ethiopia,235373.37,5.186079,4.54,-13.819384,0
1127,Viet Nam,1461716.04,2.133038,2.63,5.717973,0


,iso3,country,region,subregion,species,mean_export_t,k7_yield_temperature_r,k7_yield_precipitation_r,k8_weather_in_norm_pct,k9_temperature_anomaly_rms_c,k9_precipitation_anomaly_rms_pct,k10_export_resilience_pct,weak_harvest_years
0,BRA,Brazil,Americas,South America,arabica,1.720140e+06,-0.098502,0.120276,45.833333,0.308617,6.649922,-3.784281,7
1,VNM,Viet Nam,Asia,South-eastern Asia,robusta,1.216394e+06,-0.323331,0.205636,45.833333,0.410598,7.986432,-18.364775,2
2,COL,Colombia,Americas,South America,arabica,5.974584e+05,0.270871,-0.401170,58.333333,0.374458,5.267240,-1.016201,3
3,IDN,Indonesia,Asia,South-eastern Asia,robusta,3.874108e+05,-0.202784,-0.041729,50.000000,0.299020,10.040757,NaN,0
4,HND,Honduras,Americas,Central America,arabica,2.596477e+05,-0.070760,-0.344405,50.000000,0.445293,12.040406,-19.795641,4
5,GTM,Guatemala,Americas,Central America,arabica,2.170957e+05,-0.122997,0.211034,33.333333,0.521730,11.249197,-10.684998,2
6,UGA,Uganda,Africa,Eastern Africa,robusta,2.122408e+05,-0.030234,0.079710,50.000000,0.396362,10.430023,-12.921074,3
7,PER,Peru,Americas,South America,arabica,2.081313e+05,-0.322611,-0.076695,50.000000,0.320893,5.895755,-23.336243,5
8,IND,India,Asia,Southern Asia,robusta,1.985594e+05,-0.304719,-0.131297,54.166667,0.282711,8.157210,4.933667,1
9,ETH,Ethiopia,Africa,Eastern Africa,arabica,1.867992e+05,-0.132371,0.124646,45.833333,0.373719,9.132529,7.505450,3


Baza: data\coffee.db, 204 KB
